# Kaggle Stage-1 Exact Replication Runner (Single Notebook)

This notebook is designed for a **fresh Kaggle notebook** with only your dataset attached.

It performs end-to-end Stage-1 setup and run:
1. Clone repos.
2. Normalize paths to the author-expected layout.
3. Copy + canonicalize `.dat` files.
4. Recreate `test_set.pkl` from provided train/test loader pickles.
5. Apply a compatibility shim for public `imagen-pytorch` API differences.
6. Run Stage-1 training + evaluation with live progress and optional W&B.

Model architecture and training logic are not changed; only runtime compatibility and observability are added.


## Step 0: Fill Exactly One Placeholder

Edit only this line in the next cell:
- `DATASET_ROOT = Path("/kaggle/input/<YOUR_DATASET_SLUG_HERE>")`

Expected dataset structure:
- `/kaggle/input/<slug>/dataloaders_fixed/*.dat`
- `/kaggle/input/<slug>/split_dataloaders/train_loader.pkl`
- `/kaggle/input/<slug>/split_dataloaders/test_loader.pkl`


In [5]:
from pathlib import Path

# ========================
# REQUIRED USER INPUT
# ========================
DATASET_ROOT = Path('/kaggle/input/final-dataset-with-splits')

# ========================
# RUN CONFIG
# ========================
RUN_NAME = 'v_FC_dim64_tv4'   # new run name
EPOCHS = 100
VAL_EVERY = 4
VAL_FRAC_FROM_TRAIN = 0.20

REPO_URL = 'https://github.com/vedanggggg/cyclone-forecasting'
REPO_DIR = Path('/kaggle/working/forecast-diffmodels')

IMAGEN_REPO_URL = 'https://github.com/lucidrains/imagen-pytorch.git'
IMAGEN_PYTORCH_DIR = Path('/kaggle/working/imagen-pytorch')

ENABLE_WANDB = True
WANDB_PROJECT = "cyclone-forecasting save and run notebook"
WANDB_ENTITY = None   # or your exact wandb username slug
from kaggle_secrets import UserSecretsClient
WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
WANDB_MODE = "online" if WANDB_API_KEY else "offline"


DATA_DATALOADERS = DATASET_ROOT / 'dataloaders_fixed'
DATA_TRAIN_PKL = DATASET_ROOT / 'split_dataloaders' / 'train_loader.pkl'
DATA_TEST_PKL = DATASET_ROOT / 'split_dataloaders' / 'test_loader.pkl'

required = [DATASET_ROOT, DATA_DATALOADERS, DATA_TRAIN_PKL, DATA_TEST_PKL]
missing = [p for p in required if not p.exists()]
assert not missing, 'Missing required paths:\n' + '\n'.join(str(x) for x in missing)

print('Input validation passed.')
print('Dataset root      :', DATASET_ROOT)
print('Dat files found   :', len(list(DATA_DATALOADERS.glob('*.dat'))))
print('Train split pickle:', DATA_TRAIN_PKL)
print('Test split pickle :', DATA_TEST_PKL)


Input validation passed.
Dataset root      : /kaggle/input/final-dataset-with-splits
Dat files found   : 15
Train split pickle: /kaggle/input/final-dataset-with-splits/split_dataloaders/train_loader.pkl
Test split pickle : /kaggle/input/final-dataset-with-splits/split_dataloaders/test_loader.pkl


## Step 1: Clone Repositories


In [6]:
import subprocess

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already exists:', REPO_DIR)

if not IMAGEN_PYTORCH_DIR.exists():
    subprocess.run(['git', 'clone', IMAGEN_REPO_URL, str(IMAGEN_PYTORCH_DIR)], check=True)
else:
    print('imagen-pytorch already exists:', IMAGEN_PYTORCH_DIR)


Repo already exists: /kaggle/working/forecast-diffmodels
imagen-pytorch already exists: /kaggle/working/imagen-pytorch


## Step 2: Detect Real Project Root and Normalize Folder Names


In [7]:
import os
from pathlib import Path

# Fix accidental trailing-space folder names like "imagen "
for p in REPO_DIR.rglob('imagen '):
    target = p.parent / 'imagen'
    if not target.exists():
        os.rename(p, target)
        print('Renamed:', p, '->', target)

candidate_roots = []
for root in [REPO_DIR, *(p for p in REPO_DIR.iterdir() if p.is_dir())]:
    if (root / 'dataproc' / 'utils.py').exists() and (root / 'imagen' / 'helpers.py').exists() and (root / 'imagen' / '64_FC' / 'train64.py').exists():
        candidate_roots.append(root)

for utils_path in REPO_DIR.rglob('dataproc/utils.py'):
    root = utils_path.parent.parent
    if (root / 'imagen' / 'helpers.py').exists() and (root / 'imagen' / '64_FC' / 'train64.py').exists():
        candidate_roots.append(root)

candidate_roots = sorted(set(candidate_roots), key=lambda x: len(str(x)))
assert candidate_roots, 'Could not find project root containing dataproc/utils.py + imagen/helpers.py + imagen/64_FC/train64.py'

PROJECT_ROOT = candidate_roots[0]
DATAPROC_DIR = PROJECT_ROOT / 'dataproc'
IMAGEN_DIR = PROJECT_ROOT / 'imagen'
STAGE1_DIR = IMAGEN_DIR / '64_FC'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATAPROC_DIR =', DATAPROC_DIR)
print('IMAGEN_DIR   =', IMAGEN_DIR)
print('STAGE1_DIR   =', STAGE1_DIR)


PROJECT_ROOT = /kaggle/working/forecast-diffmodels/forecast-video-diffmodels
DATAPROC_DIR = /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/dataproc
IMAGEN_DIR   = /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/imagen
STAGE1_DIR   = /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/imagen/64_FC


## Step 3: Install Dependencies

This installs only what is needed for Stage-1 run + monitoring.


In [8]:
import sys, subprocess, shutil
from pathlib import Path

IMAGEN_PYTORCH_DIR = Path('/kaggle/working/imagen-pytorch')

# 1) Re-clone cleanly
if IMAGEN_PYTORCH_DIR.exists():
    shutil.rmtree(IMAGEN_PYTORCH_DIR)
subprocess.run(
    ['git', 'clone', 'https://github.com/lucidrains/imagen-pytorch.git', str(IMAGEN_PYTORCH_DIR)],
    check=True
)

# 2) Install with full logs (no -q)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(IMAGEN_PYTORCH_DIR)], check=True)

# 3) Install remaining deps
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'einops', 'pixelmatch', 'torchmetrics', 'lpips',
    'xarray', 'satpy', 'fsspec', 'pyproj', 'scipy', 'scikit-image',
    'openpyxl', 'dill', 'pandas', 'tensorboard', 'opencv-python', 'wandb'
], check=True)

# 4) Import check (robust for Kaggle editable installs)
import sys
if str(IMAGEN_PYTORCH_DIR) not in sys.path:
    sys.path.insert(0, str(IMAGEN_PYTORCH_DIR))

import imagen_pytorch
print('imagen_pytorch import OK:', imagen_pytorch.__file__)


import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "wandb>=0.22.3"], check=True)

import wandb
print("wandb version:", wandb.__version__)



Cloning into '/kaggle/working/imagen-pytorch'...


Obtaining file:///kaggle/working/imagen-pytorch
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'


  Building editable for imagen-pytorch (pyproject.toml): started
  Building editable for imagen-pytorch (pyproject.toml): finished with status 'done'
  Created wheel for imagen-pytorch: filename=imagen_pytorch-2.1.0-0.editable-py3-none-any.whl size=4413 sha256=1793249d1ba42425990aa6a48bbaaef851a21341814906cd29e7ce636fea62c7
  Stored in directory: /tmp/pip-ephem-wheel-cache-nwlmdh5p/wheels/64/4f/49/1568cfb45cb33b40c658af28881eff3a191ebdf39fd47bae1b
Successfully built imagen-pytorch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 36.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 102.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.4/680.4 kB 27.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 119.8 MB/s  0:00:00


2026-02-08 12:54:22.234856: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770555262.423230      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770555262.483790      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770555262.965231      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770555262.965272      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770555262.965276      55 computation_placer.cc:177] computation placer alr

...

imagen_pytorch import OK: /kaggle/working/imagen-pytorch/imagen_pytorch/__init__.py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 25.0 MB/s  0:00:00


  Attempting uninstall: wandb
    Found existing installation: wandb 0.22.2
    Uninstalling wandb-0.22.2:


      Successfully uninstalled wandb-0.22.2


wandb version: 0.24.2


## Step 4: Recreate Author-Expected `/rds/...` Paths


In [9]:
import os
import shutil

RDS_HOME = Path('/rds/general/user/zr523/home/researchProject')
RDS_DATA = Path('/rds/general/ephemeral/user/zr523/ephemeral')
RDS_PROJECT_LINK = RDS_HOME / 'forecast-diffmodels'
RDS_DATALOADER_64_FC = RDS_DATA / 'satellite' / 'dataloader' / '64_FC'

RDS_HOME.mkdir(parents=True, exist_ok=True)
RDS_DATALOADER_64_FC.mkdir(parents=True, exist_ok=True)

if RDS_PROJECT_LINK.is_symlink() or RDS_PROJECT_LINK.exists():
    if RDS_PROJECT_LINK.is_symlink() or RDS_PROJECT_LINK.is_file():
        RDS_PROJECT_LINK.unlink()
    else:
        shutil.rmtree(RDS_PROJECT_LINK)
os.symlink(str(PROJECT_ROOT), str(RDS_PROJECT_LINK))

print('Symlink:', RDS_PROJECT_LINK, '->', os.readlink(RDS_PROJECT_LINK))
print('Dataloader dir:', RDS_DATALOADER_64_FC)


Symlink: /rds/general/user/zr523/home/researchProject/forecast-diffmodels -> /kaggle/working/forecast-diffmodels/forecast-video-diffmodels
Dataloader dir: /rds/general/ephemeral/user/zr523/ephemeral/satellite/dataloader/64_FC


## Step 5: Copy and Canonicalize Dat Files

Input naming currently: `cyclone_region.dat`.

Author code expects: `region_cyclone.dat`.


In [10]:
import shutil

for old in RDS_DATALOADER_64_FC.glob('*.dat'):
    old.unlink()

copied = []
for src in sorted(DATA_DATALOADERS.glob('*.dat')):
    stem = src.stem
    assert '_' in stem, f'Unexpected filename format: {src.name}'
    cyclone, region = stem.rsplit('_', 1)
    dst = RDS_DATALOADER_64_FC / f'{region}_{cyclone}.dat'
    shutil.copy2(src, dst)
    copied.append(dst.name)

print('Copied dat files:', len(copied))
print('Sample:', copied[:6])


Copied dat files: 15
Sample: ['use_bonnie.dat', 'use_delta.dat', 'use_eta.dat', 'use_fiona.dat', 'usw_genevieve.dat', 'use_grace.dat']


## Step 6: Reconstruct `test_set.pkl` from Provided Split Pickles

This avoids assumptions and recreates the same train/test sample counts as your provided split files.


In [11]:
import pickle
from itertools import combinations

# Minimal placeholder classes for loading pickles saved from notebook context
class CycloneDataLoader: pass
class ModelDataLoader: pass

with open(DATA_TEST_PKL, 'rb') as f:
    test_loader_ref = pickle.load(f)
with open(DATA_TRAIN_PKL, 'rb') as f:
    train_loader_ref = pickle.load(f)

target_test = int(test_loader_ref.img_o.shape[0])
target_train = int(train_loader_ref.img_o.shape[0])

items = []
for fp in sorted(RDS_DATALOADER_64_FC.glob('*.dat')):
    region, name = fp.stem.split('_', 1)
    with open(fp, 'rb') as f:
        obj = pickle.load(f)
    count = int(obj.img_64.shape[0])
    items.append((region, name, count, fp))

total = sum(x[2] for x in items)
assert total == (target_train + target_test), f'Total mismatch: dat_total={total}, expected={target_train + target_test}'

idxs = list(range(len(items)))
matches = []
for r in range(1, len(items) + 1):
    for comb in combinations(idxs, r):
        test_count = sum(items[i][2] for i in comb)
        train_count = total - test_count
        if test_count == target_test and train_count == target_train:
            matches.append(comb)

assert matches, 'No cyclone subset matched provided train/test sample counts.'
matches = sorted(matches, key=lambda x: (len(x), [items[i][1] for i in x]))
chosen = matches[0]

split = {'nio': [], 'aus': [], 'wpo': [], 'wio': [], 'use': [], 'usw': []}
for i in chosen:
    region, name, _, _ = items[i]
    split[region].append(name)

if len(matches) > 1:
    print('WARNING: multiple valid test subsets found. Using smallest + lexical order by default.')
    print('Num valid subsets:', len(matches))

test_set_local = DATAPROC_DIR / 'test_set.pkl'
with open(test_set_local, 'wb') as f:
    pickle.dump(split, f)

test_set_rds = RDS_PROJECT_LINK / 'dataproc' / 'test_set.pkl'
with open(test_set_rds, 'wb') as f:
    pickle.dump(split, f)

print('test_set.pkl written:', test_set_local)
print('test split mapping :', split)
print('target train/test  :', target_train, target_test)


Num valid subsets: 18
test_set.pkl written: /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/dataproc/test_set.pkl
test split mapping : {'nio': [], 'aus': [], 'wpo': [], 'wio': [], 'use': ['delta', 'eta', 'laura'], 'usw': []}
target train/test  : 2229 816


# 6 B

In [12]:
import pickle
from itertools import combinations

# Build per-cyclone counts from canonical dat files
items = []
for fp in sorted(RDS_DATALOADER_64_FC.glob('*.dat')):
    region, name = fp.stem.split('_', 1)
    with open(fp, 'rb') as f:
        obj = pickle.load(f)
    items.append((region, name, int(obj.img_64.shape[0])))

# Load test split created in Step 6
test_set_path = DATAPROC_DIR / 'test_set.pkl'
with open(test_set_path, 'rb') as f:
    test_split = pickle.load(f)

test_pairs = {(r, n) for r, names in test_split.items() for n in names}
train_items = [x for x in items if (x[0], x[1]) not in test_pairs]

train_total = sum(x[2] for x in train_items)
target_val = int(round(train_total * VAL_FRAC_FROM_TRAIN))

idxs = list(range(len(train_items)))
best = None
for r in range(1, len(train_items) + 1):
    for comb in combinations(idxs, r):
        c = sum(train_items[i][2] for i in comb)
        score = (abs(c - target_val), len(comb), sorted(train_items[i][1] for i in comb))
        if best is None or score < best[0]:
            best = (score, comb, c)

_, val_idx, val_count = best

val_split = {'nio': [], 'aus': [], 'wpo': [], 'wio': [], 'use': [], 'usw': []}
for i in val_idx:
    r, n, _ = train_items[i]
    val_split[r].append(n)

# Fit holdout = original test + val (so training excludes both)
fit_holdout_split = {k: sorted(set(test_split.get(k, []) + val_split.get(k, [])))
                     for k in ['nio', 'aus', 'wpo', 'wio', 'use', 'usw']}

val_set_path = DATAPROC_DIR / 'val_set.pkl'
fit_holdout_path = DATAPROC_DIR / 'fit_holdout_set.pkl'

with open(val_set_path, 'wb') as f:
    pickle.dump(val_split, f)
with open(fit_holdout_path, 'wb') as f:
    pickle.dump(fit_holdout_split, f)

# Count summary
fit_holdout_pairs = {(r, n) for r, names in fit_holdout_split.items() for n in names}
fit_train_items = [x for x in items if (x[0], x[1]) not in fit_holdout_pairs]
fit_train_count = sum(x[2] for x in fit_train_items)
test_count = sum(x[2] for x in items if (x[0], x[1]) in test_pairs)

print('test split     :', test_split)
print('val split      :', val_split)
print('fit holdout    :', fit_holdout_split)
print('Counts -> train:', fit_train_count, 'val:', val_count, 'test:', test_count)
print('Files written  :', val_set_path, fit_holdout_path)


test split     : {'nio': [], 'aus': [], 'wpo': [], 'wio': [], 'use': ['delta', 'eta', 'laura'], 'usw': []}
val split      : {'nio': [], 'aus': [], 'wpo': [], 'wio': [], 'use': ['bonnie'], 'usw': ['orlene', 'roslyn']}
fit holdout    : {'nio': [], 'aus': [], 'wpo': [], 'wio': [], 'use': ['bonnie', 'delta', 'eta', 'laura'], 'usw': ['orlene', 'roslyn']}
Counts -> train: 1782 val: 447 test: 816
Files written  : /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/dataproc/val_set.pkl /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/dataproc/fit_holdout_set.pkl


## Step 7: Apply Runtime Compatibility Patch (No Architecture Change)

Public `imagen-pytorch` removed `condition_on_continuous` and `continuous_embeds` APIs.
This shim maps those to current text-conditioning internals so original script logic can run unchanged.


In [14]:
from pathlib import Path

compat_path = IMAGEN_DIR / "imagen_api_compat.py"

compat_code = r'''
import torch
from imagen_pytorch import Unet, Unet3D, Imagen as _Imagen, ImagenTrainer as _ImagenTrainer, NullUnet


def _to_text_condition(continuous_embeds):
    if continuous_embeds is None:
        return None, None

    if not torch.is_tensor(continuous_embeds):
        continuous_embeds = torch.tensor(continuous_embeds)

    if continuous_embeds.ndim == 1:
        continuous_embeds = continuous_embeds.unsqueeze(0)

    if continuous_embeds.ndim > 2:
        continuous_embeds = continuous_embeds.reshape(continuous_embeds.shape[0], -1)

    # one token per sample -> [batch, 1, dim]
    text_embeds = continuous_embeds.float().unsqueeze(1)
    text_masks = torch.ones(
        (text_embeds.shape[0], text_embeds.shape[1]),
        dtype=torch.bool,
        device=text_embeds.device
    )
    return text_embeds, text_masks


class Imagen(_Imagen):
    def __init__(self, *args, condition_on_continuous=False, continuous_embed_dim=None, **kwargs):
        if condition_on_continuous:
            kwargs.setdefault("condition_on_text", True)
            if continuous_embed_dim is not None:
                kwargs.setdefault("text_embed_dim", continuous_embed_dim)
        super().__init__(*args, **kwargs)

    def sample(self, *args, continuous_embeds=None, **kwargs):
        text_embeds, text_masks = _to_text_condition(continuous_embeds)
        if text_embeds is not None and "text_embeds" not in kwargs:
            kwargs["text_embeds"] = text_embeds
        if text_masks is not None and "text_masks" not in kwargs:
            kwargs["text_masks"] = text_masks
        return super().sample(*args, **kwargs)


class ImagenTrainer(_ImagenTrainer):
    def __call__(self, *args, continuous_embeds=None, **kwargs):
        text_embeds, text_masks = _to_text_condition(continuous_embeds)
        if text_embeds is not None and "text_embeds" not in kwargs:
            kwargs["text_embeds"] = text_embeds
        if text_masks is not None and "text_masks" not in kwargs:
            kwargs["text_masks"] = text_masks
        return super().__call__(*args, **kwargs)
'''.lstrip()

compat_path.write_text(compat_code)
print("Wrote:", compat_path)

def ensure_import_patch(path: Path, old: str, new: str):
    text = path.read_text()
    if new in text:
        return
    if old in text:
        text = text.replace(old, new)
        path.write_text(text)

utils_py = DATAPROC_DIR / "utils.py"
train_py = STAGE1_DIR / "train64.py"
eval_py = STAGE1_DIR / "v_t02-sampling-and-evaluation.py"
test_py = STAGE1_DIR / "test64.py"

old_utils = "from imagen_pytorch import Unet, Unet3D, Imagen, ImagenTrainer, NullUnet"
new_utils = (
    "try:\n"
    "    from imagen_api_compat import Unet, Unet3D, Imagen, ImagenTrainer, NullUnet\n"
    "except ImportError:\n"
    "    from imagen_pytorch import Unet, Unet3D, Imagen, ImagenTrainer, NullUnet"
)

old_train = "from imagen_pytorch import Unet3D, Imagen, ImagenTrainer"
new_train = (
    "try:\n"
    "    from imagen_api_compat import Unet3D, Imagen, ImagenTrainer\n"
    "except ImportError:\n"
    "    from imagen_pytorch import Unet3D, Imagen, ImagenTrainer"
)

old_eval_test = "from imagen_pytorch import Unet3D, Imagen, ImagenTrainer, NullUnet"
new_eval_test = (
    "try:\n"
    "    from imagen_api_compat import Unet3D, Imagen, ImagenTrainer, NullUnet\n"
    "except ImportError:\n"
    "    from imagen_pytorch import Unet3D, Imagen, ImagenTrainer, NullUnet"
)

ensure_import_patch(utils_py, old_utils, new_utils)
ensure_import_patch(train_py, old_train, new_train)
ensure_import_patch(eval_py, old_eval_test, new_eval_test)
ensure_import_patch(test_py, old_eval_test, new_eval_test)

print("Import patching done.")


Wrote: /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/imagen/imagen_api_compat.py
Import patching done.


## Step 8: Validate Imports and Dataloader Construction


In [15]:
import sys

for p in [DATAPROC_DIR, IMAGEN_DIR, IMAGEN_PYTORCH_DIR]:
    ps = str(p)
    if ps not in sys.path:
        sys.path.insert(0, ps)

import utils
from helpers import get_satellite_data

class Args: pass
args = Args()
args.batch_size = 1
args.o_size = 64
args.n_size = 128
args.dataset_path = str(RDS_DATALOADER_64_FC)
args.datalimit = False
args.mode = 'fc'
args.lr = 3e-4
args.augment = False

train_dl, test_dl = get_satellite_data(args, 'vid')
print('Import + dataloader build OK')

# v_ModelDataLoader uses img / vid fields (not img_o)
print('Train img shape :', train_dl.img.shape)
print('Test img shape  :', test_dl.img.shape)
print('Train vid shape :', train_dl.vid.shape)
print('Test vid shape  :', test_dl.vid.shape)

print('Train img count :', train_dl.img.shape[0])
print('Test img count  :', test_dl.img.shape[0])

assert train_dl.img.shape[0] == 2229, f"Unexpected train count: {train_dl.img.shape[0]}"
assert test_dl.img.shape[0] == 816, f"Unexpected test count: {test_dl.img.shape[0]}"
print('Count check passed: train=2229, test=816')


 13%|█▎        | 2/15 [00:00<00:01, 12.39it/s]

laura
fiona
ida
ian


 27%|██▋       | 4/15 [00:00<00:02,  5.45it/s]

sally


 40%|████      | 6/15 [00:01<00:02,  4.38it/s]

...

bonnie


 93%|█████████▎| 14/15 [00:05<00:00,  1.48it/s]

olaf


100%|██████████| 15/15 [00:06<00:00,  2.50it/s]

Import + dataloader build OK
Train img shape : torch.Size([2229, 64, 64])
Test img shape  : torch.Size([816, 64, 64])
Train vid shape : torch.Size([216, 10, 64, 64])
Test vid shape  : torch.Size([81, 10, 64, 64])
Train img count : 2229
Test img count  : 816
Count check passed: train=2229, test=816


In [16]:
import pickle
from pathlib import Path

items = []
for fp in sorted(RDS_DATALOADER_64_FC.glob("*.dat")):
    region, name = fp.stem.split("_", 1)
    with open(fp, "rb") as f:
        obj = pickle.load(f)
    items.append((region, name, int(obj.img_64.shape[0])))

with open(DATAPROC_DIR / "test_set.pkl", "rb") as f:
    test_split = pickle.load(f)
with open(DATAPROC_DIR / "val_set.pkl", "rb") as f:
    val_split = pickle.load(f)

test_pairs = {(r, n) for r, names in test_split.items() for n in names}
val_pairs = {(r, n) for r, names in val_split.items() for n in names}

test_count = sum(c for r, n, c in items if (r, n) in test_pairs)
val_count = sum(c for r, n, c in items if (r, n) in val_pairs)
train_count = sum(c for r, n, c in items if (r, n) not in test_pairs and (r, n) not in val_pairs)

print("train:", train_count, "val:", val_count, "test:", test_count)
print("val split:", val_split)
print("test split:", test_split)


train: 1782 val: 447 test: 816
val split: {'nio': [], 'aus': [], 'wpo': [], 'wio': [], 'use': ['bonnie'], 'usw': ['orlene', 'roslyn']}
test split: {'nio': [], 'aus': [], 'wpo': [], 'wio': [], 'use': ['delta', 'eta', 'laura'], 'usw': []}


## Step 9: Train Stage-1 with Live Progress and Optional W&B

This cell streams subprocess output and prints periodic monitor status:
- elapsed time
- latest log line
- checkpoint count
- GPU utilization


In [20]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

import os, re, glob, pickle, subprocess
from pathlib import Path

import torch
import wandb


def to_float(x):
    if torch.is_tensor(x):
        return float(x.detach().cpu().item())
    return float(x)


def average_metrics(metrics_dict, keys):
    out = {}
    for k in keys:
        vals = metrics_dict.get(k, [])
        if len(vals) == 0:
            continue
        out[k] = sum(to_float(v) for v in vals) / len(vals)
    return out


def run_cmd_live(cmd, cwd, env):
    print("[run]", " ".join(cmd))
    p = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


fit_holdout_path = DATAPROC_DIR / "fit_holdout_set.pkl"
val_set_path = DATAPROC_DIR / "val_set.pkl"
test_set_path = DATAPROC_DIR / "test_set.pkl"

assert fit_holdout_path.exists(), f"Missing {fit_holdout_path}"
assert val_set_path.exists(), f"Missing {val_set_path}"
assert test_set_path.exists(), f"Missing {test_set_path}"

# Shared env
base_env = os.environ.copy()
base_env["PYTHONPATH"] = f"{DATAPROC_DIR}:{IMAGEN_DIR}:{IMAGEN_PYTORCH_DIR}:{base_env.get('PYTHONPATH','')}"
base_env["PYTHONUNBUFFERED"] = "1"
base_env["FDM_PROJECT_ROOT"] = str(PROJECT_ROOT)
base_env["FDM_BASE_HOME"] = str(RDS_PROJECT_LINK)
base_env["FDM_BASE_DATA"] = str(RDS_DATA)
base_env["FDM_DATAPROC_DIR"] = str(DATAPROC_DIR)
base_env["FDM_IMAGEN_DIR"] = str(IMAGEN_DIR)
base_env["FDM_TEST_SET_PATH"] = str(test_set_path)  # default, overridden per phase

print(
    "split files -> "
    f"fit_holdout: {fit_holdout_path} | "
    f"val: {val_set_path} | "
    f"test: {test_set_path}"
)

# --- non-interactive W&B auth for Save & Run ---
WB_MODE = "disabled"
if ENABLE_WANDB:
    key = (WANDB_API_KEY or "").strip()
    if key:
        base_env["WANDB_API_KEY"] = key
        base_env["WANDB_SILENT"] = "true"
        base_env["WANDB_CONSOLE"] = "off"

        os.environ["WANDB_API_KEY"] = key
        os.environ["WANDB_SILENT"] = "true"
        os.environ["WANDB_CONSOLE"] = "off"

        try:
            wandb.login(key=key, relogin=True)
            WB_MODE = WANDB_MODE if WANDB_MODE in ("online", "offline", "disabled") else "online"
            print(f"[wandb] login ok, mode={WB_MODE}")
        except Exception as e:
            WB_MODE = "offline"
            print(f"[wandb] login failed ({e}); falling back to offline mode")
    else:
        WB_MODE = "disabled"
        print("[wandb] WANDB_API_KEY missing; wandb disabled")

# One W&B run for validation curves across all checkpoints
val_run = None
if ENABLE_WANDB and WB_MODE != "disabled":
    val_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=f"{RUN_NAME}-val",
        mode=WB_MODE,
        config={
            "run_name": RUN_NAME,
            "epochs": EPOCHS,
            "val_every": VAL_EVERY,
            "val_metrics": ["mae", "psnr", "ssim", "fid", "fvd", "rmse", "lpips"]
        }
    )

metric_keys = ["mae", "psnr", "ssim", "fid", "fvd", "rmse", "lpips"]

for target_epoch in range(VAL_EVERY, EPOCHS + 1, VAL_EVERY):
    print(f"\n===== TRAIN TO EPOCH {target_epoch} =====")

    # Train split: exclude val + test
    env_train = base_env.copy()
    env_train["FDM_TEST_SET_PATH"] = str(fit_holdout_path)

    train_cmd = [
        "python", "-u", "train64.py",
        "-mode", "experiment",              # saves per epoch
        "-run_name", RUN_NAME,
        "-epochs", str(target_epoch),
        "--progress_interval", "25"
    ]

    if ENABLE_WANDB and WB_MODE != "disabled":
        train_cmd += [
            "--enable_wandb",
            "--wandb_project", WANDB_PROJECT,
            "--wandb_mode", WB_MODE,
            "--wandb_run_name", f"{RUN_NAME}-train"
        ]
        if WANDB_ENTITY:
            train_cmd += ["--wandb_entity", WANDB_ENTITY]

    run_cmd_live(train_cmd, STAGE1_DIR, env_train)

    # Pick latest checkpoint by numeric epoch suffix
    ckpt_dir = RDS_HOME / "models" / RUN_NAME / "models" / RUN_NAME
    ckpts = glob.glob(str(ckpt_dir / "ckpt_trainer_1_*.pt"))
    assert ckpts, f"No ckpt_trainer files found in {ckpt_dir}"

    def ckpt_epoch_num(p):
        m = re.search(r"_(\d{3})\.pt$", Path(p).name)
        return int(m.group(1)) if m else -1

    ckpts = sorted(ckpts, key=ckpt_epoch_num)
    latest = Path(ckpts[-1]).name
    ckpt_epoch = ckpt_epoch_num(ckpts[-1])
    assert ckpt_epoch >= 0, f"Could not parse epoch from {latest}"
    print(f"Validating checkpoint epoch index: {ckpt_epoch} ({latest})")

    # Validation split only
    env_val = base_env.copy()
    env_val["FDM_TEST_SET_PATH"] = str(val_set_path)

    val_cmd = [
        "python", "-u", "test64.py",
        "-run_name", RUN_NAME,
        "-best_epoch", str(ckpt_epoch)
    ]
    run_cmd_live(val_cmd, STAGE1_DIR, env_val)

    metrics_pkl = RDS_HOME / "models" / RUN_NAME / "metrics_test.pkl"
    assert metrics_pkl.exists(), f"Missing {metrics_pkl}"
    with open(metrics_pkl, "rb") as f:
        metric_dict = pickle.load(f)

    avg = average_metrics(metric_dict, metric_keys)
    print(f"[VAL @ epoch {target_epoch}] {avg}")

    if val_run is not None:
        log_payload = {"val/ckpt_epoch_index": ckpt_epoch}
        log_payload.update({f"val/{k}": v for k, v in avg.items()})
        val_run.log(log_payload, step=target_epoch)

if val_run is not None:
    val_run.finish()

print("Train+val loop complete.")


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


split files -> fit_holdout: /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/dataproc/fit_holdout_set.pkl | val: /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/dataproc/val_set.pkl | test: /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/dataproc/test_set.pkl
[wandb] login ok, mode=online



===== TRAIN TO EPOCH 4 =====
[run] python -u train64.py -mode experiment -run_name v_FC_dim64_tv4 -epochs 4 --progress_interval 25 --enable_wandb --wandb_project cyclone-forecasting save and run notebook --wandb_mode online --wandb_run_name v_FC_dim64_tv4-train
2026-02-08 13:04:38.313631: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770555878.336778     257 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770555878.344051     257 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770555878.362033     257 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

CalledProcessError: Command '['python', '-u', 'train64.py', '-mode', 'experiment', '-run_name', 'v_FC_dim64_tv4', '-epochs', '4', '--progress_interval', '25', '--enable_wandb', '--wandb_project', 'cyclone-forecasting save and run notebook', '--wandb_mode', 'online', '--wandb_run_name', 'v_FC_dim64_tv4-train']' returned non-zero exit status 1.

In [ ]:
import os, glob, re, pickle, subprocess
from pathlib import Path
import torch

def to_float(x):
    if torch.is_tensor(x):
        return float(x.detach().cpu().item())
    return float(x)

def average_metrics(metrics_dict, keys):
    out = {}
    for k in keys:
        vals = metrics_dict.get(k, [])
        if len(vals) == 0:
            continue
        out[k] = sum(to_float(v) for v in vals) / len(vals)
    return out

test_set_path = DATAPROC_DIR / 'test_set.pkl'
env = os.environ.copy()
env['PYTHONPATH'] = f"{DATAPROC_DIR}:{IMAGEN_DIR}:{IMAGEN_PYTORCH_DIR}:{env.get('PYTHONPATH','')}"
env['PYTHONUNBUFFERED'] = '1'
env['FDM_PROJECT_ROOT'] = str(PROJECT_ROOT)
env['FDM_BASE_HOME'] = str(RDS_PROJECT_LINK)
env['FDM_BASE_DATA'] = str(RDS_DATA)
env['FDM_DATAPROC_DIR'] = str(DATAPROC_DIR)
env['FDM_IMAGEN_DIR'] = str(IMAGEN_DIR)
env['FDM_TEST_SET_PATH'] = str(test_set_path)

if ENABLE_WANDB and WANDB_API_KEY:
    env['WANDB_API_KEY'] = WANDB_API_KEY

ckpt_dir = RDS_HOME / 'models' / RUN_NAME / 'models' / RUN_NAME
ckpts = glob.glob(str(ckpt_dir / 'ckpt_trainer_1_*.pt'))
assert ckpts, f'No checkpoints found in {ckpt_dir}'

def epoch_from_path(p):
    m = re.search(r'_(\d{3})\.pt$', Path(p).name)
    assert m, f'Could not parse epoch from {p}'
    return int(m.group(1))

ckpts = sorted(ckpts, key=epoch_from_path)
latest = Path(ckpts[-1]).name
best_epoch = epoch_from_path(ckpts[-1])
print('Final test checkpoint epoch index:', best_epoch, '| file:', latest)

subprocess.run(
    ['python', '-u', 'test64.py', '-run_name', RUN_NAME, '-best_epoch', str(best_epoch)],
    cwd=str(STAGE1_DIR),
    env=env,
    check=True
)

metrics_pkl = RDS_HOME / 'models' / RUN_NAME / 'metrics_test.pkl'
assert metrics_pkl.exists(), f'Missing metrics file: {metrics_pkl}'

with open(metrics_pkl, 'rb') as f:
    metric_dict = pickle.load(f)

keys = ['mae', 'psnr', 'ssim', 'fid', 'fvd', 'rmse', 'lpips']
final_avg = average_metrics(metric_dict, keys)
print('Final TEST averages:', final_avg)

if ENABLE_WANDB:
    import wandb
    run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=f'{RUN_NAME}-test-final',
        mode=WANDB_MODE,
        reinit=True
    )
    run.log({f'test/{k}': v for k, v in final_avg.items()})
    run.finish()
    print('Logged final test metrics to W&B.')


## Step 12: Output Locations


In [ ]:
from pathlib import Path

model_root = Path('/rds/general/user/zr523/home/researchProject/models') / RUN_NAME
print('Run log               :', model_root / 'run.log')
print('Model checkpoints     :', model_root / 'models' / RUN_NAME)
print('Result images         :', model_root / 'results' / RUN_NAME)
print('Validation metrics pkl:', model_root / 'metrics_test.pkl')  # test64 writes here
print('Eval sweep metrics pkl:', model_root / 'metrics.pkl')       # only if v_t02 script is run
print('test_set.pkl          :', DATAPROC_DIR / 'test_set.pkl')
print('val_set.pkl           :', DATAPROC_DIR / 'val_set.pkl')
print('fit_holdout_set.pkl   :', DATAPROC_DIR / 'fit_holdout_set.pkl')
